In [2]:
"""
MODULE 6a — Unit Conversion & Price Normalisation
===================================================
INPUT  : Dataset/Exports/**/*.csv  (raw files, same as Modules 1-5)
OUTPUT : Dataset/Normalised/prices_normalised.csv

What this module does:
  1. Loads all raw CSV files
  2. Converts every Quantity to a common base unit per physical dimension:
       Weight  -> kg      Volume -> litre     Count -> count
  3. Computes normalised unit price = Value ($) / normalised_qty
  4. Extracts season (Q1-Q4) from the monthly period string
  5. Saves a clean normalised file for Module 6b and 6c to consume
  6. Prints a verification table so you can spot-check conversions are correct

Run this first, then run Module 6b (outlier detection).
"""

import glob
import pandas as pd
import numpy as np
from pathlib import Path

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
BASE_DIR = Path('/content/drive/MyDrive/Dashboard-BR-CA-Data')
EXPORTS_DIR  = BASE_DIR / 'Exports'
OUTPUT_DIR    = BASE_DIR / "Dataset" / "Normalised"
OUTPUT_FILE   = OUTPUT_DIR / "prices_normalised.csv"

# Rows shown per unit label in the verification table
VERIFY_SAMPLE = 3

MONTH_TO_SEASON: dict[int, str] = {
    1: "Q1", 2: "Q1", 3: "Q1",
    4: "Q2", 5: "Q2", 6: "Q2",
    7: "Q3", 8: "Q3", 9: "Q3",
    10: "Q4", 11: "Q4", 12: "Q4",
}

# Official unit labels exactly as they appear in the dataset
UNIT_CONVERSION: dict[str, tuple[str, float]] = {
    # Weight -> kg
    "Weight in grams":                        ("kg",    0.001),
    "Weight in kilograms":                    ("kg",    1.0),
    "Weight in metric tonne":                 ("kg",    1_000.0),
    "Metric tonne air dry":                   ("kg",    1_000.0),
    "Weight in kilograms of named substance": ("kg",    1.0),
    "Weight in carats":                       ("kg",    0.0002),
    # Volume -> litre
    "Volume in litres":                       ("litre", 1.0),
    "Volume in cubic metres":                 ("litre", 1_000.0),
    "Volume in litres of pure alcohol":       ("litre", 1.0),
    "Thousands of cubic metres":              ("litre", 1_000_000.0),
    # Count -> count
    "Number":                                 ("count", 1.0),
    "Number of pairs":                        ("count", 1.0),
    "Number of dozens":                       ("count", 12.0),
    "Number of packages":                     ("count", 1.0),
    "Number of gross":                        ("count", 144.0),
    "Number in thousands":                    ("count", 1_000.0),
    # Area -> m2  (flooring, carpets, leather hides, etc.)
    "Area in square metres":                  ("m2",    1.0),
    # Length -> m  (mouldings, dowels, rods)
    "Length in metres":                       ("m",     1.0),
    # Energy -> MWh  (electrical energy exports, HS 2716)
    "Megawatt-Hour":                          ("MWh",   1.0),
}

# "Blank" is a special Statistics Canada marker — it means no supplementary
# quantity was recorded for this HS code. Quantity is always 0 in these rows.
# Value ($) still exists, but unit price cannot be computed.
# This is NOT a data error — it is by design for certain commodity codes.
BLANK_UNIT_LABEL = "Blank"

RADIOACTIVITY_CONVERSION: dict[str, tuple[str, float]] = {
    "Radioactivity in megabecquerels": ("GBq", 0.001),
    "Radioactivity in gigabecquerels": ("GBq", 1.0),
}

_BASE_UNIT_DIMENSION: dict[str, str] = {
    "kg":    "weight",
    "litre": "volume",
    "count": "count",
    "m2":    "area",
    "m":     "length",
    "MWh":   "energy",
    "GBq":   "radioactivity",
}


# ──────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────
def _parse_period(series: pd.Series) -> pd.Series:
    """
    Parse period strings to datetime.
    The dataset uses YYYY-MM-DD (e.g. "2024-11-01"), NOT YYYY-MM.
    errors="coerce" also silently drops the 3-line Statistics Canada
    citation footer at the end of each file (those rows have non-date
    text in the Period column and parse to NaT).
    """
    return pd.to_datetime(series, errors="coerce")


def load_raw(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath, skiprows=1, dtype=str)
    df.columns = df.columns.str.strip()
    if "Period" in df.columns:
        parsed = _parse_period(df["Period"].astype(str).str.strip())
        # Keep only rows where Period is a valid date — this also removes
        # the Statistics Canada footer rows at the bottom of each file
        df = df[parsed.notna()].copy()
    df["_source_file"] = Path(filepath).name
    return df


def load_all(files: list) -> tuple[pd.DataFrame, list]:
    frames, errors = [], []
    for f in files:
        try:
            frames.append(load_raw(f))
        except Exception as e:
            errors.append({"file": Path(f).name, "error": str(e)})
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return combined, errors


# ──────────────────────────────────────────────────────────────
# STEP 1 — UNIT CONVERSION
# ──────────────────────────────────────────────────────────────
def convert_and_price(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds columns:
      _base_unit          : canonical unit after conversion (kg/litre/count/m2/m/MWh/GBq)
      _conversion_factor  : multiplier applied to raw Quantity
      _qty_normalised     : raw Quantity x factor
      _unit_price         : Value ($) / _qty_normalised
      _dimension          : weight/volume/count/area/length/energy/radioactivity/blank/unknown
      _conversion_status  : OK / BLANK_UNIT / RADIOACTIVITY / UNKNOWN_UNIT / NO_PRICE / ZERO_QTY
      _season             : Q1-Q4 derived from Period month
      _hs_chapter         : 2-digit HS chapter prefix (e.g. "08" for fruit)
      _seasonal_risk_flag : True if commodity is in a seasonally-priced HS chapter

    Conversion status values:
      OK           — unit price computed successfully
      BLANK_UNIT   — Statistics Canada "Blank" unit; quantity is always 0 by design;
                     value exists but no unit price can be computed; NOT a data error
      RADIOACTIVITY— radioactivity unit (GBq); excluded from price outlier analysis
      UNKNOWN_UNIT — unit label not in our table; add it to UNIT_CONVERSION above
      NO_PRICE     — Value ($) is missing
      ZERO_QTY     — normalised quantity is zero or negative after conversion
    """
    df = df.copy()
    raw_qty  = pd.to_numeric(df.get("Quantity"), errors="coerce")
    value    = pd.to_numeric(df.get("Value ($)"), errors="coerce")
    unit_col = (
        df.get("Unit of measure", pd.Series("", index=df.index))
          .fillna("").astype(str).str.strip()
    )

    base_units, factors, dimensions, statuses = [], [], [], []

    for u in unit_col:
        if u == BLANK_UNIT_LABEL:
            # "Blank" is a valid Statistics Canada code meaning no supplementary
            # quantity exists for this commodity. Qty is always 0. Not an error.
            base_units.append("blank"); factors.append(np.nan)
            dimensions.append("blank"); statuses.append("BLANK_UNIT")
        elif u in UNIT_CONVERSION:
            base, factor = UNIT_CONVERSION[u]
            base_units.append(base); factors.append(factor)
            dimensions.append(_BASE_UNIT_DIMENSION.get(base, "unknown"))
            statuses.append("OK")
        elif u in RADIOACTIVITY_CONVERSION:
            base, factor = RADIOACTIVITY_CONVERSION[u]
            base_units.append(base); factors.append(factor)
            dimensions.append("radioactivity"); statuses.append("RADIOACTIVITY")
        else:
            base_units.append(u or "[missing]"); factors.append(np.nan)
            dimensions.append("unknown"); statuses.append("UNKNOWN_UNIT")

    df["_base_unit"]         = base_units
    df["_conversion_factor"] = pd.array(factors, dtype=float)
    df["_dimension"]         = dimensions
    df["_conversion_status"] = statuses

    factor_s              = pd.Series(factors, index=df.index, dtype=float)
    df["_qty_normalised"] = raw_qty * factor_s
    df["_value"]          = value

    # Status refinement for OK rows only
    ok_mask  = df["_conversion_status"] == "OK"
    no_price = value.isna()
    zero_qty = df["_qty_normalised"].notna() & (df["_qty_normalised"] <= 0)
    df.loc[ok_mask & no_price,              "_conversion_status"] = "NO_PRICE"
    df.loc[ok_mask & (~no_price) & zero_qty,"_conversion_status"] = "ZERO_QTY"

    computable = (
        (df["_conversion_status"] == "OK")
        & value.notna()
        & df["_qty_normalised"].notna()
        & (df["_qty_normalised"] > 0)
    )
    df["_unit_price"] = np.nan
    df.loc[computable, "_unit_price"] = value[computable] / df.loc[computable, "_qty_normalised"]

    # Season from YYYY-MM-DD period
    parsed = _parse_period(
        df.get("Period", pd.Series("", index=df.index))
          .fillna("").astype(str).str.strip()
    )
    df["_month"]  = parsed.dt.month
    df["_year"]   = parsed.dt.year
    df["_season"] = df["_month"].map(MONTH_TO_SEASON).fillna("Unknown")

    # ── Seasonal risk: HS chapter prefix ──────────────────────
    # Commodity names look like "0808.10.90 - Apples, fresh, other than for processing"
    # The first 2 characters are the HS chapter — the international standard for
    # trade goods classification. This is far more reliable than English keyword
    # matching, which would never match against the full HS trade descriptions.
    #
    # Chapters flagged as seasonal (prices naturally vary by time of year):
    #   01-05  Live animals, meat, fish, dairy, eggs
    #   06-14  Plants, vegetables, fruit, coffee, cereals, oil seeds
    #   15-16  Fats/oils, prepared meat/fish
    #   22     Beverages (fruit wines, ciders — harvest-driven)
    #   27     Mineral fuels, petroleum (energy seasonality)
    SEASONAL_HS_CHAPTERS = {
        "01","02","03","04","05",
        "06","07","08","09","10",
        "11","12","13","14","15",
        "16","22","27",
    }
    commodity_col = df.get("Commodity", pd.Series("", index=df.index)).fillna("").astype(str)
    df["_hs_chapter"]         = commodity_col.str[:2]
    df["_seasonal_risk_flag"] = df["_hs_chapter"].isin(SEASONAL_HS_CHAPTERS)

    return df




# ──────────────────────────────────────────────────────────────
# STEP 2 — CROSS-DIMENSION CHECK
# ──────────────────────────────────────────────────────────────
def cross_dimension_report(df: pd.DataFrame) -> pd.DataFrame:
    """Commodities appearing under more than one physical dimension."""
    issues = (
        df[df["_dimension"] != "unknown"]
        .groupby("Commodity")["_dimension"]
        .nunique()
        .reset_index(name="n_dim")
    )
    issues = issues[issues["n_dim"] > 1]
    if issues.empty:
        return pd.DataFrame()

    rows = []
    for commodity in issues["Commodity"]:
        sub = df[df["Commodity"] == commodity]
        breakdown = "  |  ".join(
            f"{r['_dimension']} ({r['rows']:,} rows)"
            for _, r in (
                sub.groupby("_dimension").size()
                   .reset_index(name="rows")
                   .sort_values("rows", ascending=False)
            ).iterrows()
        )
        rows.append({"Commodity": commodity,
                     "n_dimensions": int(issues.loc[issues["Commodity"] == commodity, "n_dim"].values[0]),
                     "breakdown": breakdown})
    return pd.DataFrame(rows).sort_values("n_dimensions", ascending=False)


# ──────────────────────────────────────────────────────────────
# STEP 3 — CONVERSION VERIFICATION TABLE
# ──────────────────────────────────────────────────────────────
def build_verification_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each unique (Unit of measure, base_unit, conversion_factor) combination,
    shows VERIFY_SAMPLE example rows with raw qty, normalised qty, value, and
    computed unit price. Lets you confirm 'Weight in metric tonne' -> kg is
    producing sensible $/kg values alongside 'Weight in kilograms' rows.
    """
    rows = []
    groups = df.groupby(
        ["Unit of measure", "_base_unit", "_conversion_factor"],
        dropna=False
    )

    for (unit_label, base_unit, factor), group in groups:
        sample = group.dropna(subset=["_unit_price"]).head(VERIFY_SAMPLE)
        for _, r in sample.iterrows():
            rows.append({
                "Unit of measure":   unit_label,
                "Base unit":         base_unit,
                "Factor":            factor,
                "Commodity":         r.get("Commodity", "?"),
                "Period":            r.get("Period", "?"),
                "Raw Quantity":      r.get("Quantity", "?"),
                "Normalised Qty":    round(r["_qty_normalised"], 4) if pd.notna(r.get("_qty_normalised")) else "?",
                "Value ($)":         r.get("Value ($)", "?"),
                "Unit Price ($/base)": round(r["_unit_price"], 6) if pd.notna(r.get("_unit_price")) else "?",
            })

    return pd.DataFrame(rows)


# ──────────────────────────────────────────────────────────────
# PRINT + SAVE
# ──────────────────────────────────────────────────────────────
def print_and_save(df_norm: pd.DataFrame,
                   cross_dim: pd.DataFrame,
                   verify: pd.DataFrame,
                   load_errors: list) -> None:

    S1 = "=" * 90
    S2 = "-" * 90

    print("\n" + S1)
    print("  MODULE 6a - UNIT CONVERSION & PRICE NORMALISATION")
    print(S1)

    # -- Conversion status summary ----------------------------
    print("\n" + S2)
    print("  1. CONVERSION STATUS SUMMARY")
    print(S2 + "\n")

    status_counts = df_norm["_conversion_status"].value_counts()
    total = len(df_norm)
    labels = {
        "OK":            "OK — unit price computed",
        "BLANK_UNIT":    "BLANK_UNIT — no qty by design (Statistics Canada 'Blank')",
        "RADIOACTIVITY": "EXCLUDED — radioactivity unit (GBq)",
        "UNKNOWN_UNIT":  "WARN — unit label not in conversion table",
        "NO_PRICE":      "WARN — Value ($) missing",
        "ZERO_QTY":      "WARN — zero or negative quantity after conversion",
    }
    for status, cnt in status_counts.items():
        label = labels.get(status, status)
        print(f"   {label:<55} : {cnt:>7,}  ({cnt/total*100:.1f}%)")

    computable = (df_norm["_conversion_status"] == "OK") & df_norm["_unit_price"].notna()
    n_seas = df_norm.get("_seasonal_risk_flag", pd.Series(False, index=df_norm.index)).sum()
    print(f"\n   {'Rows with a valid unit price':<55} : {computable.sum():>7,}  "
          f"({computable.sum()/total*100:.1f}%)")
    print(f"   {'Rows in seasonal HS chapters (01-16, 22, 27)':<55} : {n_seas:>7,}  "
          f"({n_seas/total*100:.1f}%)")

    # -- Unknown units ----------------------------------------
    unknown_mask = df_norm["_conversion_status"] == "UNKNOWN_UNIT"
    if unknown_mask.any():
        print("\n" + S2)
        print("  2. UNKNOWN UNIT LABELS  (add to UNIT_CONVERSION in CONFIG)")
        print(S2 + "\n")
        unk_counts = (
            df_norm.loc[unknown_mask, "Unit of measure"]
            .value_counts()
            .reset_index()
            .rename(columns={"Unit of measure": "Unit label", "count": "Rows"})
        )
        for _, r in unk_counts.iterrows():
            print(f"   {str(r.iloc[0]):<55} {int(r.iloc[1]):>6,} rows")
        print()

    # -- Cross-dimension issues --------------------------------
    print("\n" + S2)
    print("  3. CROSS-DIMENSION MISMATCHES")
    print("     (same commodity under both weight and volume, etc.)")
    print(S2 + "\n")
    if cross_dim.empty:
        print("   OK  No cross-dimension mismatches detected.\n")
    else:
        for _, r in cross_dim.iterrows():
            print(f"   WARN  {r['Commodity']}")
            print(f"         {r['breakdown']}\n")

    # -- Verification table -----------------------------------
    print("\n" + S2)
    print(f"  4. CONVERSION VERIFICATION TABLE  ({VERIFY_SAMPLE} examples per unit label)")
    print("     Check that Normalised Qty and Unit Price look correct for each unit.")
    print(S2 + "\n")

    if verify.empty:
        print("   No rows with valid unit prices to verify.\n")
    else:
        # Group by unit label for readable output
        for unit_label, grp in verify.groupby("Unit of measure"):
            base = grp["Base unit"].iloc[0]
            factor = grp["Factor"].iloc[0]
            print(f"   {unit_label}  ->  {base}  (x {factor})")
            print(f"   {'Commodity':<35} {'Period':<10} {'Raw Qty':>12} "
                  f"{'Norm Qty (' + base + ')':>18} {'Value ($)':>12} "
                  f"{'$/'+base:>14}")
            print("   " + "-" * 86)
            for _, r in grp.iterrows():
                print(f"   {str(r['Commodity']):<35} {str(r['Period']):<10} "
                      f"{str(r['Raw Quantity']):>12} "
                      f"{str(r['Normalised Qty']):>18} "
                      f"{str(r['Value ($)']):>12} "
                      f"{str(r['Unit Price ($/base)']):>14}")
            print()

    # -- Load errors ------------------------------------------
    if load_errors:
        print("\n" + S2)
        print("  5. FILE LOAD ERRORS")
        print(S2 + "\n")
        for e in load_errors:
            print(f"   {e['file']} : {e['error']}")
        print()


def save_output(df_norm: pd.DataFrame) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    save_cols = [c for c in [
        "Period", "_year", "_month", "_season",
        "Commodity", "Province", "Country", "State",
        "Value ($)", "Quantity", "Unit of measure",
        "_base_unit", "_conversion_factor", "_qty_normalised",
        "_unit_price", "_dimension", "_conversion_status",
        "_hs_chapter", "_seasonal_risk_flag", "_source_file",
    ] if c in df_norm.columns]

    df_norm[save_cols].to_csv(OUTPUT_FILE, index=False)
    print(f"\n   Saved: {OUTPUT_FILE}")
    print(f"   Total rows saved : {len(df_norm):,}")
    valid = df_norm["_unit_price"].notna().sum()
    print(f"   Valid unit prices: {valid:,}  (others have NaN _unit_price)")


# ──────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":

    print("\n" + "=" * 90)
    print("  MODULE 6a - UNIT CONVERSION & PRICE NORMALISATION")
    print("=" * 90)

    export_files = glob.glob(str(EXPORTS_DIR / "**/*.csv"), recursive=True)
    print(f"\nFound {len(export_files)} CSV file(s) in {EXPORTS_DIR}")

    if not export_files:
        print("No files found. Check EXPORTS_DIR path.")
        exit(1)

    print("Loading files...")
    raw, load_errors = load_all(export_files)
    print(f"Loaded {len(raw):,} rows from {len(set(raw['_source_file']))} file(s)")

    print("Converting units and computing prices...")
    df_norm = convert_and_price(raw)

    cross_dim = cross_dimension_report(df_norm)
    verify    = build_verification_table(df_norm)

    print_and_save(df_norm, cross_dim, verify, load_errors)

    print("\nSaving normalised file...")
    save_output(df_norm)

    print("\n" + "=" * 90)
    print("  DONE — run Module 6b next for outlier detection.")
    print("=" * 90 + "\n")



  MODULE 6a - UNIT CONVERSION & PRICE NORMALISATION

Found 158 CSV file(s) in /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/Exports
Loading files...
Loaded 2,562,365 rows from 158 file(s)
Converting units and computing prices...

  MODULE 6a - UNIT CONVERSION & PRICE NORMALISATION

------------------------------------------------------------------------------------------
  1. CONVERSION STATUS SUMMARY
------------------------------------------------------------------------------------------

   OK — unit price computed                                : 1,288,311  (50.3%)
   BLANK_UNIT — no qty by design (Statistics Canada 'Blank') : 1,259,828  (49.2%)
   WARN — zero or negative quantity after conversion       :  10,843  (0.4%)
   EXCLUDED — radioactivity unit (GBq)                     :   3,383  (0.1%)

   Rows with a valid unit price                            : 1,288,311  (50.3%)
   Rows in seasonal HS chapters (01-16, 22, 27)            : 285,170  (11.1%)

---------------